In [ ]:
# installing sempy labs
# https://semantic-link-labs.readthedocs.io/en/latest/modules.html
# https://github.com/microsoft/semantic-link-labs
# https://learn.microsoft.com/en-us/fabric/data-science/semantic-link-overview

%pip install semantic-link-labs semantic-link --upgrade

## Initialization

In [ ]:
import sempy.fabric as fabric
import sempy_labs as labs
from sempy_labs.tom import connect_semantic_model
import warnings
import pandas as pd

client = fabric.PowerBIRestClient()

workspace = "your-workspace-name"
dataset = "dataset | semantic model name"

workspace_id = fabric.resolve_workspace_id(workspace)
dataset_id = fabric.resolve_dataset_id(dataset, workspace_id)

print(f"""
workspace name: {workspace}
Workspace id: {workspace_id}
dataset: {dataset}
dataset id: {dataset_id}
""")

## Checking role membership of the model

In [ ]:
# Retrieve user memberships
members = client.get(f"v1.0/myorg/datasets/{dataset_id}/users").json()
for m in members["value"]:
    print(m)

In [ ]:
roles_data = []
table_permissions_data = []
members_data = []

with connect_semantic_model(dataset=dataset, workspace=workspace_id) as tom:
    for role in tom.model.Roles:
        # Role-level info
        roles_data.append(
            {
                "RoleName": role.Name,
                "ModelPermission": str(role.ModelPermission),
            }
        )

        # Table permissions for this role
        for tp in role.TablePermissions:
            table_permissions_data.append(
                {
                    "RoleName": role.Name,
                    "TableName": tp.Table.Name,
                    "FilterExpression": tp.FilterExpression or "(none)",
                }
            )

        # Members for this role
        for member in role.Members:
            members_data.append(
                {
                    "RoleName": role.Name,
                    "MemberName": member.Name,
                    "MemberTypeClass": type(member).__name__,
                    "MemberType": str(member.MemberType),
                }
            )

df_roles = pd.DataFrame(roles_data)
df_table_permissions = pd.DataFrame(table_permissions_data)
df_members = pd.DataFrame(members_data)

display(df_roles)
display(df_table_permissions)
display(df_members)


### Detecting orphanned security groups in RLS

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 1: Read all members from production model
# ══════════════════════════════════════════════════════════

from sempy_labs.tom import connect_semantic_model
import Microsoft.AnalysisServices.Tabular as TOM
import sempy_labs as labs
import json

test_model = "___rls_validation_temp"

all_members = []
with connect_semantic_model(dataset=dataset, workspace=workspace, readonly=True) as tom:
    for role in tom.model.Roles:
        for member in role.Members:
            all_members.append({
                "role":             role.Name,
                "memberName":       member.MemberName,
                "memberId":         member.MemberID,
                "memberType":       str(member.MemberType),
                "identityProvider": member.IdentityProvider
            })

print(f"Total members to validate: {len(all_members)}")

In [ ]:
# ══════════════════════════════════════════════════════════
# CELL 2: Validate every member against Fabric's own validator
# ══════════════════════════════════════════════════════════

labs.create_blank_semantic_model(dataset=test_model, workspace=workspace)

with connect_semantic_model(dataset=test_model, workspace=workspace, readonly=False) as tom:
    role = TOM.ModelRole()
    role.Name = "ValidationRole"
    role.ModelPermission = TOM.ModelPermission.Read
    tom.model.Roles.Add(role)

orphaned = []
valid    = []

for m in all_members:
    try:
        with connect_semantic_model(dataset=test_model, workspace=workspace, readonly=False) as tom:
            role = tom.model.Roles["ValidationRole"]
            role.Members.Clear()

            member = TOM.ExternalModelRoleMember()
            member.MemberName       = m["memberName"]
            member.MemberID         = m["memberId"]
            member.IdentityProvider = m["identityProvider"]
            if "Group" in m["memberType"]:
                member.MemberType = TOM.RoleMemberType.Group

            role.Members.Add(member)
        valid.append(m)
        print(f"  ✅ {m['memberType']} | {m['memberName']}")
    except:
        orphaned.append(m)
        print(f"  ❌ {m['memberType']} | {m['memberName']}")

print(f"\n✅ Valid: {len(valid)}")
print(f"❌ Orphaned: {len(orphaned)}")

# Clean up
labs.delete_semantic_model(dataset=test_model, workspace=workspace)